# Feature Ablation: annual_submission_rate

## Purpose

`annual_submission_rate` is a derived feature, defined as `total_submissions / submission_history_years`. Because it is constructed from two features that are themselves in the model, a natural question is whether it carries independent predictive signal, or whether it merely restates information already captured by its two components.

This notebook answers that question empirically. The model is retrained with and without `annual_submission_rate` on an identical train/test split, and Average Precision (AP) is compared. If removing the feature leaves AP essentially unchanged, the feature is redundant and can be dropped for a more parsimonious model. If AP falls materially, the feature contributes signal that its components do not, and it is retained.

## Why the model is held fixed

A feature ablation isolates the effect of one feature, so the model is fixed to XGBoost, the configuration selected in the main comparison, and only the feature set is varied. Any change in AP is then attributable to the feature rather than to a change of algorithm. (The separate question of whether financial ratios add value is addressed across all five models in the financial-value notebook, because there the unit of comparison is a feature class, not a single feature.)

## Method

Single XGBoost fit per configuration on the existing train/test sets, fixed hyperparameters held constant across both arms. A threshold of 0.003 AP is treated as the boundary of practical significance.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
import xgboost as xgb

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import PROCESSED_FILES, RANDOM_STATE

train = pd.read_csv(PROCESSED_FILES['train_set'], low_memory=False)
test  = pd.read_csv(PROCESSED_FILES['test_set'],  low_memory=False)
print(f'Train {len(train):,} | Test {len(test):,}')
assert 'annual_submission_rate' in train.columns, 'feature not found in train_set'

Train 98,926 | Test 94,421


In [2]:
feat_path = ROOT / 'models' / 'feature_cols.txt'
if feat_path.exists():
    FEATURES = [l.strip() for l in open(feat_path) if l.strip()]
else:
    non_feat = {'company_num','company_name','company_status','company_type','comp_reg_date',
        'comp_dissolved_date','last_ar_date','last_accounts_date','nace_v2_code',
        'company_address_1','company_address_2','company_address_3','company_address_4','eircode',
        'obs_date','county','nace_sector','company_type_norm','nace_section','nace_section_label',
        'label','dissolution_risk_score','dissolution_risk_tier','if_anomaly_score',
        'if_anomaly_flag','combined_risk_score','combined_risk_tier'}
    FEATURES = [c for c in train.columns if c not in non_feat and train[c].dtype != object]

y_tr = train['label'].values.astype(int)
y_te = test['label'].values.astype(int)
pos_weight = (y_tr==0).sum()/max((y_tr==1).sum(),1)
print(f'{len(FEATURES)} features | scale_pos_weight = {pos_weight:.2f}')

84 features | scale_pos_weight = 13.94


In [3]:
def fit_eval(features, label):
    Xtr = train[features].fillna(0).values
    Xte = test[features].fillna(0).values
    m = xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
        scale_pos_weight=pos_weight, subsample=0.8, colsample_bytree=0.8,
        eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(Xtr, y_tr)
    p = m.predict_proba(Xte)[:,1]
    ap, auc = average_precision_score(y_te,p), roc_auc_score(y_te,p)
    print(f'  {label:<30} AP={ap:.4f}  AUC={auc:.4f}  ({len(features)} features)')
    return ap, auc

print('='*60)
ap_with, _    = fit_eval(FEATURES, 'WITH annual_submission_rate')
without       = [f for f in FEATURES if f != 'annual_submission_rate']
ap_without, _ = fit_eval(without, 'WITHOUT annual_submission_rate')
print('='*60)

  WITH annual_submission_rate    AP=0.5636  AUC=0.9341  (84 features)
  WITHOUT annual_submission_rate AP=0.5404  AUC=0.9304  (83 features)


In [4]:
d_ap = ap_with - ap_without
print(f'AP with feature:    {ap_with:.4f}')
print(f'AP without feature: {ap_without:.4f}')
print(f'AP delta:           {d_ap:+.4f}')
print()
if d_ap <= 0.003:
    print('VERDICT: removal costs <= 0.003 AP. The feature is redundant given its')
    print('components and can be dropped for a more parsimonious model.')
else:
    print(f'VERDICT: removal costs {d_ap:.4f} AP, above the 0.003 threshold.')
    print('The feature carries independent predictive signal not captured by its')
    print('components, and is retained.')

AP with feature:    0.5636
AP without feature: 0.5404
AP delta:           +0.0232

VERDICT: removal costs 0.0232 AP, above the 0.003 threshold.
The feature carries independent predictive signal not captured by its
components, and is retained.


## Interpretation

If dropping `annual_submission_rate` reduces test AP by more than the 0.003 threshold, the feature contributes signal beyond what `total_submissions` and `submission_history_years` provide individually: the *intensity* of filing (volume normalised by tenure) is informative in a way that neither raw count nor raw tenure captures alone, and the feature is retained.

On the question of collinearity: the feature is by construction correlated with its two components. This is immaterial for a gradient-boosted tree model, which is robust to correlated inputs, since each split selects whichever single feature is most useful at that node. Collinearity would matter for a linear model interpreted through its coefficients, but the deployed model is XGBoost, interpreted through SHAP attributions, for which correlated features are handled without instability.